In [2]:
from pathlib import Path
import json
import math
import warnings

import joblib
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

SOURCE_DIR = Path('/Users/marclamberts/Event data/Ecuador 2026')
DEST_DIR = Path('/Users/marclamberts/Event data/Ecuador 2026/Danger')
MODEL_DIR = Path('/Users/marclamberts/Downloads/Danger Model/xg_output')

DEST_DIR.mkdir(parents=True, exist_ok=True)

# This class must exist before joblib.load() because the models were pickled with it in __main__.
class CalibratedXGB:
    def __init__(self, clf=None, iso=None, clip=(0.005, 0.95)):
        self.clf = clf
        self.iso = iso
        self.clip = clip

    def predict_proba(self, X):
        raw = self.clf.predict_proba(X)[:, 1]
        calibrated = self.iso.predict(raw)
        if hasattr(self, 'clip') and self.clip is not None:
            calibrated = np.clip(calibrated, *self.clip)
        return np.column_stack([1 - calibrated, calibrated])


def load_pickle(name):
    path = MODEL_DIR / name
    if not path.exists():
        raise FileNotFoundError(path)
    return joblib.load(path)


meta = load_pickle('model_meta.pkl')
psxg_meta = load_pickle('model_psxg_meta.pkl')

models = {
    'xg': load_pickle('model_xg.pkl'),
    'psxg': load_pickle('model_psxg.pkl'),
    'situation_danger': load_pickle('model_situation.pkl'),
    'p_on_target': load_pickle('model_pontarget.pkl'),
    'xgot': load_pickle('model_xgot.pkl'),
    'multi_outcome': load_pickle('model_multi_outcome.pkl'),
}

XG_FEATURES = list(meta['features'])
PSXG_FEATURES = list(psxg_meta['features'])
SIT_FEATURES = list(meta['sit_features'])
POT_FEATURES = list(meta['pot_features'])
PEN_XG = float(meta.get('pen_xg', psxg_meta.get('pen_xg', 0.79)))
Q = dict(meta.get('qualifier_mapping', psxg_meta.get('qualifier_mapping', {})))

GOAL_X = 100.0
GOAL_Y_CENTER = 50.0
GOAL_Y_LEFT = 44.0
GOAL_Y_RIGHT = 56.0
GOAL_WIDTH = 12.0
SHOT_TYPE_IDS = {13, 14, 15, 16}


def qid(name, default=None):
    return int(Q.get(name, default)) if Q.get(name, default) is not None else None


def has_q(event, qualifier_id):
    if qualifier_id is None:
        return False
    return any(int(q.get('qualifierId', -1)) == int(qualifier_id) for q in event.get('qualifier', []))


def get_q(event, qualifier_id):
    if qualifier_id is None:
        return None
    for q in event.get('qualifier', []):
        if int(q.get('qualifierId', -1)) == int(qualifier_id):
            return q.get('value', 1)
    return None


def safe_float(value):
    try:
        return float(value)
    except Exception:
        return np.nan


def shot_angle(x, y):
    shot = np.array([x, y], dtype=float)
    left = np.array([GOAL_X, GOAL_Y_LEFT], dtype=float)
    right = np.array([GOAL_X, GOAL_Y_RIGHT], dtype=float)
    v1 = left - shot
    v2 = right - shot
    denom = np.linalg.norm(v1) * np.linalg.norm(v2)
    if denom == 0:
        return 0.0
    return float(np.degrees(np.arccos(np.clip(np.dot(v1, v2) / denom, -1, 1))))


def placement_score(goal_y_norm, goal_h_norm):
    if pd.isna(goal_y_norm) or pd.isna(goal_h_norm):
        return np.nan
    lateral = abs(float(np.clip(goal_y_norm, -1, 1)))
    vertical = abs(float(np.clip(goal_h_norm, 0, 1)) - 0.40) / 0.60
    return float(np.clip(np.sqrt(0.6 * lateral**2 + 0.4 * vertical**2) * 100, 0, 100))


def load_events(path):
    with path.open('r', encoding='utf-8-sig') as f:
        payload = json.load(f)
    if isinstance(payload, dict):
        return payload.get('liveData', {}).get('event', payload.get('event', []))
    return []


def parse_shot(event, match_file):
    try:
        type_id = int(event.get('typeId'))
    except Exception:
        return None
    if type_id not in SHOT_TYPE_IDS:
        return None

    x = safe_float(event.get('x'))
    y = safe_float(event.get('y'))
    if pd.isna(x) or pd.isna(y):
        return None

    distance = math.hypot(GOAL_X - x, y - GOAL_Y_CENTER)
    angle = shot_angle(x, y)
    y_sym = abs(y - GOAL_Y_CENTER)

    goal_y_raw = safe_float(get_q(event, qid('GOAL_Y')))
    goal_h_raw = safe_float(get_q(event, qid('GOAL_HEIGHT')))
    goal_y_norm = (goal_y_raw - GOAL_Y_CENTER) / (GOAL_WIDTH / 2) if not pd.isna(goal_y_raw) else np.nan
    goal_h_norm = goal_h_raw / 100.0 if not pd.isna(goal_h_raw) else np.nan
    goal_frame_dist = (
        math.sqrt(goal_y_norm**2 + (goal_h_norm - 0.35)**2)
        if not pd.isna(goal_y_norm) and not pd.isna(goal_h_norm)
        else np.nan
    )

    right_foot = has_q(event, qid('RIGHT_FOOT'))
    left_foot = has_q(event, qid('LEFT_FOOT'))
    header = has_q(event, qid('HEADER'))
    other_body = has_q(event, qid('OTHER_BODY_PART')) or not (right_foot or left_foot or header)

    blocked_by_outfielder = type_id == 15 and has_q(event, qid('BLOCKED'))
    keeper_saved_off_target = type_id == 15 and has_q(event, qid('KEEPER_SAVED_OFF_TARGET'))
    on_target = type_id == 16 or (type_id == 15 and not blocked_by_outfielder and not keeper_saved_off_target)

    assisted = has_q(event, qid('ASSISTED')) or has_q(event, qid('RELATED_EVENT_ID'))
    intentional_assist = has_q(event, qid('INTENTIONAL_ASSIST'))
    individual_play = has_q(event, qid('INDIVIDUAL_PLAY')) or not assisted

    row = {
        'match_file': match_file,
        'event_id': event.get('id', event.get('eventId')),
        'event_index': event.get('eventId'),
        'period_id': int(event.get('periodId') or 0),
        'time_min': int(event.get('timeMin') or 0),
        'time_sec': int(event.get('timeSec') or 0),
        'contestant_id': event.get('contestantId'),
        'player_id': event.get('playerId'),
        'player_name': event.get('playerName', ''),
        'type_id': type_id,
        'x': x,
        'y': y,
        'distance': distance,
        'angle': angle,
        'y_sym': y_sym,
        'goal_y_raw': goal_y_raw,
        'goal_h_raw': goal_h_raw,
        'goal_y_norm': goal_y_norm,
        'goal_h_norm': goal_h_norm,
        'goal_frame_dist': goal_frame_dist,
        'corner_zone': int(abs(goal_y_norm) > 0.55 or goal_h_norm > 0.60) if not pd.isna(goal_frame_dist) else 0,
        'placement_score': placement_score(goal_y_norm, goal_h_norm),
        'is_goal': int(type_id == 16),
        'is_post': int(type_id == 14),
        'is_on_target': int(on_target),
        'is_outfield_block': int(blocked_by_outfielder),
        'is_header': int(header),
        'is_right_foot': int(right_foot),
        'is_left_foot': int(left_foot),
        'is_other_body_part': int(other_body),
        'is_volley': int(has_q(event, qid('VOLLEY')) or has_q(event, qid('HALF_VOLLEY'))),
        'is_one_on_one': int(has_q(event, qid('ONE_ON_ONE'))),
        'is_fast_break': int(has_q(event, qid('FAST_BREAK'))),
        'is_from_corner': int(has_q(event, qid('FROM_CORNER'))),
        'is_corner_second_phase': int(has_q(event, qid('CORNER_SECOND_PHASE'))),
        'is_free_kick': int(has_q(event, qid('DIRECT_FREE_KICK'))),
        'is_penalty': int(has_q(event, qid('PENALTY'))),
        'is_set_piece': int(has_q(event, qid('SET_PIECE'))),
        'is_throw_in_set_piece': int(has_q(event, qid('THROW_IN_SET_PIECE'))),
        'is_open_play': int(has_q(event, qid('REGULAR_PLAY'))),
        'is_assisted': int(assisted),
        'is_intentional_assist': int(intentional_assist),
        'is_individual_play': int(individual_play),
        'is_big_chance': int(has_q(event, qid('BIG_CHANCE'))),
        'is_scramble': int(has_q(event, qid('SCRAMBLE'))),
        'is_deflected': int(has_q(event, qid('DEFLECTION'))),
    }
    return row


def ensure_features(df, features):
    df = df.copy()
    for feature in features:
        if feature not in df.columns:
            df[feature] = 0
    return df[features].fillna(0).astype(float)


def predict_binary(model, df, features):
    if len(df) == 0:
        return np.array([], dtype=float)
    return model.predict_proba(ensure_features(df, features))[:, 1]


def add_model_outputs(shots):
    shots = shots.copy()
    for col in sorted(set(XG_FEATURES + PSXG_FEATURES + SIT_FEATURES + POT_FEATURES)):
        if col not in shots.columns:
            shots[col] = 0

    shots['xg'] = np.nan
    shots['psxg'] = np.nan
    shots['situation_danger'] = np.nan
    shots['p_on_target'] = np.nan
    shots['xgot'] = np.nan
    shots['multi_miss'] = np.nan
    shots['multi_post'] = np.nan
    shots['multi_on_target'] = np.nan
    shots['multi_goal'] = np.nan

    penalty_mask = shots['is_penalty'].astype(bool)
    nonpen_mask = ~penalty_mask

    if penalty_mask.any():
        shots.loc[penalty_mask, ['xg', 'psxg', 'xgot', 'situation_danger']] = PEN_XG
        shots.loc[penalty_mask, 'p_on_target'] = 1.0
        shots.loc[penalty_mask, ['multi_miss', 'multi_post', 'multi_on_target']] = 0.0
        shots.loc[penalty_mask, 'multi_goal'] = PEN_XG

    if nonpen_mask.any():
        nonpen = shots.loc[nonpen_mask].copy()
        shots.loc[nonpen_mask, 'xg'] = predict_binary(models['xg'], nonpen, XG_FEATURES)
        shots.loc[nonpen_mask, 'psxg'] = predict_binary(models['psxg'], nonpen, PSXG_FEATURES)
        shots.loc[nonpen_mask, 'situation_danger'] = predict_binary(models['situation_danger'], nonpen, SIT_FEATURES)
        shots.loc[nonpen_mask, 'p_on_target'] = predict_binary(models['p_on_target'], nonpen, POT_FEATURES)

        placed_mask = nonpen[['goal_y_norm', 'goal_h_norm']].notna().any(axis=1)
        if placed_mask.any():
            placed = nonpen.loc[placed_mask].copy()
            shots.loc[placed.index, 'xgot'] = predict_binary(models['xgot'], placed, PSXG_FEATURES)
        shots.loc[nonpen.index[~placed_mask], 'xgot'] = shots.loc[nonpen.index[~placed_mask], 'psxg']

        multi_proba = models['multi_outcome'].predict_proba(ensure_features(nonpen, XG_FEATURES))
        multi_cols = ['multi_miss', 'multi_post', 'multi_on_target', 'multi_goal']
        for i, col in enumerate(multi_cols[:multi_proba.shape[1]]):
            shots.loc[nonpen_mask, col] = multi_proba[:, i]

    shots['danger_score'] = shots[['xg', 'psxg', 'situation_danger', 'p_on_target', 'xgot']].mean(axis=1)
    return shots


def process_file(path):
    rows = [parse_shot(event, path.name) for event in load_events(path)]
    shots = pd.DataFrame([row for row in rows if row is not None])
    if shots.empty:
        print(f'{path.name}: no shots found')
        return shots

    shots = add_model_outputs(shots)
    output_cols = [
        'match_file', 'event_id', 'event_index', 'period_id', 'time_min', 'time_sec',
        'contestant_id', 'player_id', 'player_name', 'type_id', 'x', 'y',
        'is_goal', 'is_post', 'is_on_target', 'is_outfield_block', 'is_penalty',
        'xg', 'psxg', 'situation_danger', 'p_on_target', 'xgot',
        'multi_miss', 'multi_post', 'multi_on_target', 'multi_goal', 'danger_score',
    ]
    output_cols = [col for col in output_cols if col in shots.columns]
    out = shots[output_cols].copy()
    out_path = DEST_DIR / f'{path.stem}_danger_models.csv'
    out.to_csv(out_path, index=False)
    print(f'{path.name}: {len(out)} shots -> {out_path.name}')
    return out


json_files = sorted(p for p in SOURCE_DIR.glob('*.json') if p.is_file())
if not json_files:
    raise FileNotFoundError(f'No JSON files found in {SOURCE_DIR}')

all_outputs = []
for json_path in json_files:
    result = process_file(json_path)
    if not result.empty:
        all_outputs.append(result)

if all_outputs:
    combined = pd.concat(all_outputs, ignore_index=True)
    combined_path = DEST_DIR / 'all_eredivisie_danger_models.csv'
    combined.to_csv(combined_path, index=False)
    print(f'Combined: {len(combined)} shots across {len(all_outputs)} files -> {combined_path}')
else:
    print('No shot rows were written.')


2026-02-20_Orense SC - Liga Deportiva Universitaria de Quito.json: 26 shots -> 2026-02-20_Orense SC - Liga Deportiva Universitaria de Quito_danger_models.csv
2026-02-21_Barcelona SC - CD Técnico Universitario.json: 16 shots -> 2026-02-21_Barcelona SC - CD Técnico Universitario_danger_models.csv
2026-02-21_SD Aucas - Mushuc Runa SC.json: 22 shots -> 2026-02-21_SD Aucas - Mushuc Runa SC_danger_models.csv
2026-02-22_CSD Independiente del Valle - Guayaquil City FC.json: 28 shots -> 2026-02-22_CSD Independiente del Valle - Guayaquil City FC_danger_models.csv
2026-02-22_CSD Macará - Leones FC.json: 24 shots -> 2026-02-22_CSD Macará - Leones FC_danger_models.csv
2026-02-22_Delfín SC - CD Cuenca.json: 22 shots -> 2026-02-22_Delfín SC - CD Cuenca_danger_models.csv
2026-02-23_Libertad Fútbol Club - Manta FC.json: 31 shots -> 2026-02-23_Libertad Fútbol Club - Manta FC_danger_models.csv
2026-02-27_CD Técnico Universitario - Orense SC.json: 26 shots -> 2026-02-27_CD Técnico Universitario - Orense S